# Fine-Tune Tutorial: Extraction, Scoring, Blending, GGUF

This notebook is a hands-on tutorial for your workflow:
1. Extract data from Claude raw logs and agentMemory DB
2. Build success-only tool-call datasets
3. Score trajectories and generate preference pairs
4. Blend multiple datasets with explicit weights
5. Validate formatting for SFT / chat / preference
6. Prepare a GGUF-compatible LoRA pipeline for LM Studio / llama.cpp

Before running, install optional fine-tune deps in a separate env:
`bash fine-tune/install_finetune_env.sh && source .venv-finetune/bin/activate`

All raw data stays under `data/raw/`.
All processed data stays under `data/processed/`.


In [ ]:
from pathlib import Path
import json
import subprocess
import textwrap

ROOT = Path.cwd()
print('ROOT =', ROOT)
print('has fine-tune scripts:', (ROOT / 'fine-tune').exists())


def run(cmd: str, check: bool = True):
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout.strip())
    if p.stderr:
        print(p.stderr.strip())
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return p


def read_json(path: str):
    return json.loads(Path(path).read_text())


def preview_jsonl(path: str, n: int = 2):
    p = Path(path)
    print(f"\nPreview: {p}")
    with p.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line[:2000].rstrip())


## Step 1: Collect Raw Logs (All Claude + Anvil)

This copies all discoverable logs into `data/raw/` so downstream steps are reproducible.


In [ ]:
run('bash fine-tune/collect_raw_data.sh')
run("find data/raw/claude -type f -name '*.jsonl' | wc -l")
run("find data/raw/anvil -type f -name '*.json' | wc -l")


## Step 2: Inspect Raw Claude Log Structure

Claude JSONL logs are event streams, not simple chat rows. This is why we use `prepare_from_claude_dir.py`.


In [ ]:
first = run("find data/raw/claude -type f -name '*.jsonl' | head -n1", check=True).stdout.strip()
print('first file:', first)
run(f"sed -n '1,5p' '{first}'")


## Step 3: Build Dataset from All Claude Logs

This yields a generic conversational dataset (`claude_all`).


In [ ]:
run('./.venv/bin/python fine-tune/prepare_from_claude_dir.py --input-dir data/raw/claude --output-dir data/processed/claude_all')
stats_claude = read_json('data/processed/claude_all/stats.json')
stats_claude


In [ ]:
preview_jsonl('data/processed/claude_all/train.chat.jsonl', n=3)
preview_jsonl('data/processed/claude_all/train.instruction_response.jsonl', n=3)


## Step 4: Export Success-Only Tool Calls from agentMemory DB

Use the DB export as your highest-signal tool-intelligence data source.


In [ ]:
# Global success-only export (all projects)
run('./.venv/bin/python fine-tune/export_from_agent_memory.py --dataset-type sft --limit 10000 --output-dir data/raw/agent_memory_global')
latest_global = run("find data/raw/agent_memory_global -type f -name 'sft_*.jsonl' | sort | tail -n1").stdout.strip()
print('latest_global:', latest_global)

run(f"./.venv/bin/python fine-tune/prepare_jsonl.py --input '{latest_global}' --input-format agent_memory --output-dir data/processed/fine_tune_global")
stats_global = read_json('data/processed/fine_tune_global/stats.json')
stats_global


## Step 5: Focus Exports for fire-map + DailyDispatch

This gives stronger domain weighting for your main ecosystems.


In [ ]:
run('./.venv/bin/python fine-tune/export_from_agent_memory.py --dataset-type sft --project /Users/mz/Dropbox/_CODING/fire-map.wfca.com/wfca-app --strict-project --limit 10000 --output-dir data/raw/agent_memory_focus')
run('./.venv/bin/python fine-tune/export_from_agent_memory.py --dataset-type sft --project /Users/mz/Dropbox/_CODING/DailyDispatch.local --strict-project --limit 10000 --output-dir data/raw/agent_memory_focus')

fire = run("find data/raw/agent_memory_focus -type f -name 'sft_*fire-map*jsonl' | sort | tail -n1").stdout.strip()
dd = run("find data/raw/agent_memory_focus -type f -name 'sft_*dailydispatch*jsonl' | sort | tail -n1").stdout.strip()
print('fire_map file:', fire)
print('daily_dispatch file:', dd)

run(f"./.venv/bin/python fine-tune/prepare_jsonl.py --input '{fire}' --input-format agent_memory --output-dir data/processed/fine_tune_focus_firemap")
run(f"./.venv/bin/python fine-tune/prepare_jsonl.py --input '{dd}' --input-format agent_memory --output-dir data/processed/fine_tune_focus_dailydispatch")

read_json('data/processed/fine_tune_focus_firemap/stats.json'), read_json('data/processed/fine_tune_focus_dailydispatch/stats.json')


## Step 6: Blend Datasets with Weights

Recommended starter blend:
- Global tool-calls: weight `2`
- fire-map tool-calls: weight `3`
- DailyDispatch tool-calls: weight `3`
- All-Claude conversational data: weight `1`


In [ ]:
run('./.venv/bin/python fine-tune/blend_chat_datasets.py '     '--source data/processed/fine_tune_global/train.chat.jsonl:2 '     '--source data/processed/fine_tune_focus_firemap/train.chat.jsonl:3 '     '--source data/processed/fine_tune_focus_dailydispatch/train.chat.jsonl:3 '     '--source data/processed/claude_all/train.chat.jsonl:1 '     '--output-dir data/processed/fine_tune_blend')

blend_stats = read_json('data/processed/fine_tune_blend/stats.json')
blend_stats


In [ ]:
preview_jsonl('data/processed/fine_tune_blend/train.chat.jsonl', n=3)
preview_jsonl('data/processed/fine_tune_blend/train.instruction_response.jsonl', n=3)


## Step 7: Score Multi-Step Trajectories (Reward Model Signals)

This score emphasizes:
- successful tool chains
- QA/test signals
- docs updates
- planning/review evidence

Tune weights in `fine-tune/rl_reward_profile.json`.


In [ ]:
run('./.venv/bin/python fine-tune/build_success_trajectories.py --limit 12000 --successful-only --profile fine-tune/rl_reward_profile.json --raw-output-dir data/raw/rl --processed-output-dir data/processed/rl')
latest_scored = run("find data/processed/rl -type f -name 'rl_episodes_scored_*.jsonl' | sort | tail -n1").stdout.strip()
print('latest_scored:', latest_scored)
preview_jsonl(latest_scored, n=2)


In [ ]:
# quick reward distribution
import statistics
rewards = []
with open(latest_scored, 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        rewards.append(obj.get('reward', 0.0))

print('episodes:', len(rewards))
if rewards:
    print('min:', min(rewards), 'max:', max(rewards), 'mean:', round(statistics.mean(rewards), 4))


## Step 8: Build Preference Pairs for DPO/ORPO

Pairs are built from high-reward vs low-reward trajectories.


In [ ]:
run(f"./.venv/bin/python fine-tune/continuous_rl_loop.py --episodes '{latest_scored}' --high-reward 2.0 --low-reward 1.0 --output-dir data/processed/rl")
latest_pairs = run("find data/processed/rl -type f -name 'preference_pairs_*.jsonl' | sort | tail -n1").stdout.strip()
print('latest_pairs:', latest_pairs)
preview_jsonl(latest_pairs, n=2)


## Step 9: Formatting Best Practices (What to Train On)

Use this policy for your first iterations:

1. Primary SFT corpus: `data/processed/fine_tune_blend/train.chat.jsonl`
2. Keep tool-call-heavy data dominant (global + focused sources)
3. Keep generic conversational Claude data lower weight
4. Keep preference pairs for a second-stage alignment run
5. Re-score and re-blend after each evaluation cycle


## Step 10: GGUF-Compatible Fine-Tune Path

Run HF LoRA -> merge -> GGUF convert/quantize.


In [ ]:
# 10a) generate HF LoRA trainer script (dry-run)
run('./.venv/bin/python fine-tune/gguf/train_lora_hf.py '     '--base-model google/gemma-2-2b-it '     '--train-file data/processed/fine_tune_blend/train.chat.jsonl '     '--valid-file data/processed/fine_tune_blend/valid.chat.jsonl '     '--output-dir models/lora/gemma2-toolcalls-lora')

run("sed -n '1,80p' models/lora/gemma2-toolcalls-lora/run_train_lora.py")


In [ ]:
# 10b) after training, merge + convert to GGUF (commands preview)
run('./.venv/bin/python fine-tune/gguf/convert_to_gguf.py '     '--hf-model-dir models/merged/gemma2-toolcalls-merged '     '--out-f16 models/gguf/gemma2-toolcalls-f16.gguf '     '--out-quant models/gguf/gemma2-toolcalls-q4km.gguf')


## Iteration Checklist

For each training iteration:
1. Re-run Steps 4–8 with fresh DB/log data
2. Inspect stats and sample rows
3. Adjust blend weights and reward profile
4. Retrain adapter and validate in Anvil
5. Convert best checkpoint to GGUF for LM Studio / llama.cpp testing
